In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import yaml
from tqdm.auto import tqdm


plt.rcParams["figure.dpi"] = 300

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())


METHOD = CFG["vpr"]["method"]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
THRESHOLD = CFG["retrieval"]["threshold"]

PROJECT_ROOT = find_project_root()
RESULT_DIR = PROJECT_ROOT / "results" 
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
RETRIEVAL_DIR = RESULT_DIR / "retrieval"
embedding_path = EMBEDDING_DIR / f"{METHOD}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{METHOD}_metadata.parquet"

RETRIEVAL_DIR.mkdir(parents = True, exist_ok = True)

embedding_metadata = pd.read_parquet(metadata_path)
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()
database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)

embeddings = np.load(embedding_path)

database_embeddings = embeddings[database_mask]
query_embeddings = embeddings[query_mask]


retrieval = np.load(RETRIEVAL_DIR / METHOD / f"{METHOD}_retrieval.npz")
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]


In [ ]:
# harvisine distance
# https://en.wikipedia.org/wiki/Haversine_formula

def haversine_distance(
        lat1,
        lon1,
        lat2,
        lon2
):
    earth_radius = 6_371_000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = (np.sin(diff_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(diff_lon / 2) ** 2)

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c



In [ ]:
def ground_truth(query_index, query_metadata, database_metadata, threshold = THRESHOLD):
    query = query_metadata.iloc[query_index]

    haversine_distances = haversine_distance(
        query["lat"],
        query["lon"],
        database_metadata["lat"].to_numpy(),
        database_metadata["lon"].to_numpy()
    )
    ground = np.where(haversine_distances <= threshold)[0]

    return ground, haversine_distances

In [ ]:
def recall_k( retrieved_indices, ground):

    return int(np.isin(retrieved_indices, ground).any())

In [ ]:
K_VALUES = [1,5,10,20]

recall_results = { k: [] for k in K_VALUES }

for query_index in tqdm(range(len(query_embeddings)),desc = "Calculating Recall"):
    ground, haversine_distances = ground_truth(query_index, query_metadata, database_metadata, threshold= THRESHOLD)

    if len(ground) == 0:
        continue

    query_retrieved_indices = retrieved_indices[query_index]


    for k in K_VALUES:
        recall = recall_k(query_retrieved_indices[:k], ground)
        recall_results[k].append(recall)


for k in K_VALUES:
    recall = np.mean(recall_results[k])
    print(f"Recall@{k}: {recall:.2f}")


ground, distances = ground_truth(0, query_metadata, database_metadata, threshold=float(THRESHOLD))

print("Min distance:", distances.min())
print("Max distance:", distances.max())
print("Ground truth:", ground)

In [ ]:
query_index = 0

ground, distances = ground_truth(
    query_index,
    query_metadata,
    database_metadata,
    threshold=THRESHOLD,
)

top_indices = retrieved_indices[query_index][:20]

print("Query:", query_metadata.iloc[query_index]["image_id"])
print()

for rank, idx in enumerate(top_indices, start=1):
    print(
        f"Rank {rank:2d} | "
        f"DB index {idx:5d} | "
        f"distance = {distances[idx]:8.2f} m | "
        f"image = {database_metadata.iloc[idx]['image_id']}"
    )
